# Reference-run workflow: exact audit of `jld2_pulse_loader.jl`

This notebook calls the **same functions** `optimise_control_pulse_from_jld2` sequences, in the same order, with the same defaults.

1. Open a `*.jld2` file.
2. Extract `SYSTEM_CONFIG`, `PULSE_CONFIG`, `SIM_SETTING`.
3. Parse them — these are the **reference configs**.
4. If `PULSE_CONFIG` is missing or unparseable, load the sibling `*_pulsemat.csv`.
5. Forward-simulate the recorded drive (`:ground` and `:equator`). Those metrics are the **reference metrics**.
6. If the file stores results, reconcile this run's final-state outputs and metrics against them. If it stores none, auto-PASS.
7. On PASS, linear-fit (`fit_mode=:linear`) a seed from `PULSE_CONFIG` (or the CSV).
8. `optimise_composite_pulse` (`pulse_optimizer2.jl`) on the same reference ensemble, warm-started from that seed.

**Kernel:** Julia, with this repo as the active project (`Pkg.activate` below).


In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

using InhomogeneousSpinCavityDynamics
using Printf


## 1–4. Open, extract, parse (CSV only as fallback)

Knobs come from `jld2_pipeline_defaults()` / `jld2_optimizer_defaults()` so this notebook cannot drift from `optimise_control_pulse_from_jld2`.


In [ ]:
pipe = jld2_pipeline_defaults()
opt  = jld2_optimizer_defaults()

JLD2_PATH = joinpath(@__DIR__, "..", "data", "data_1st_order", "3ARP_pi_gstd_1em06Hz.jld2")
isfile(JLD2_PATH) || error("Reference file not found: $JLD2_PATH")

println("JLD2_PATH = $JLD2_PATH")
println()
println("=== jld2_pipeline_defaults() ===")
for (k, v) in pairs(pipe)
    @printf("  %-20s = %s\n", k, repr(v))
end
println()
println("=== jld2_optimizer_defaults() ===")
for (k, v) in pairs(opt)
    @printf("  %-22s = %s\n", k, repr(v))
end


In [ ]:
ref = load_jld2_reference(
    JLD2_PATH;
    n_signal  = pipe.n_signal,
    use_signal = pipe.use_signal,
    fit_N     = pipe.fit_N,
    verbose   = true,
)

data = ref.data
d    = ref.d
RELTOL = data.SIM_SETTING.reltol
ABSTOL = data.SIM_SETTING.abstol


Parsed reference configs: cavity / ensemble / pulse source.


In [ ]:
println("=== Cavity ===")
@printf("  kappa_e (external) = %.6g\n", d.kappa_e)
@printf("  kappa_i (internal) = %.6g\n", d.kappa_i)
@printf("  kappa_t (total)    = %.6g\n", d.kappa_t)
@printf("  delta0             = %.6g\n", d.delta0)

println()
println("=== Ensemble ===")
@printf("  C_ens (cooperativity)     = %.6g\n", d.C_ens)
@printf("  N (total spin number)     = %.6g\n", d.N)
@printf("  N_total (binned check)    = %.6g\n", d.N_total)
@printf("  M_delta x M_g -> M        = %d x %d -> %d bins\n", d.M_delta, d.M_g, d.M)
@printf("  FWHM (frequency inhom.)   = %.6g\n", d.FWHM)
@printf("  g_mean / g_std (coupling) = %.6g / %.6g\n", d.g_mean, d.g_std)
@printf("  timespan                  = (%.6g, %.6g) s\n", d.timespan[1], d.timespan[2])

println()
println("=== Pulse source ===")
println("  parse_ok     = $(ref.parse_ok)")
println("  parse message= $(ref.parse_message)")
println("  pulse source = $(ref.pulse_source)")
println("  reason       = $(ref.pulse_source_reason)")
println("  USE_SIGNAL   = $(ref.use_signal)")
println("  USE_SIGNAL note = $(ref.use_signal_note)")

println()
if ref.signal_cfg === nothing
    println("=== Signal pulse(s) ===")
    println("  <none — PULSE_CONFIG did not parse; signal drive is _zero_drive>")
else
    println("=== Signal pulse(s) ($(length(ref.signal_cfg))) ===")
    for (i, cfg) in enumerate(ref.signal_cfg)
        println("  [$i] kind=$(cfg.kind)  $cfg")
    end
end

println()
if ref.control_cfg === nothing
    println("=== Control pulse(s) ===")
    println("  <none as PULSE_CONFIG specs; control I/Q has $(length(ref.control_t)) samples from $(ref.control_trace_note)>")
else
    println("=== Control pulse(s) ($(length(ref.control_cfg))) ===")
    for (i, cfg) in enumerate(ref.control_cfg)
        println("  [$i] kind=$(cfg.kind)  $cfg")
    end
end


## 5. Forward simulation (`:ground` and `:equator`)

`run_reference_forward` — the same pair of final-state solves `optimise_control_pulse_from_jld2` uses. The returned metrics are the **reference metrics**.


In [ ]:
forward = run_reference_forward(
    ref; reltol=RELTOL, abstol=ABSTOL, compute=opt.compute, verbose=true,
)
reference_metrics = forward.metrics

println()
println("=== Reference metrics ===")
@printf("  inversion = %.6g\n", reference_metrics.inversion)
@printf("  silencing = %.6g\n", reference_metrics.silencing)
@printf("  coherence = %.6g\n", reference_metrics.coherence)
@printf("  duration  = %.6g s\n", reference_metrics.duration)


## 6. Reconcile against stored results (final state only)

If the `.jld2` stores `a_sol`/`Σp_sol`/`Σz_sol` and/or named metrics, compare this run against them. If it stores none, auto-PASS.


In [ ]:
ok, report = reconcile_reference(
    ref, forward; rtol=pipe.rtol_check, atol=pipe.atol_check, verbose=true,
)
println("status: ", ok ? "PASS" : "FAIL", report.auto_pass ? " (auto-PASS: no stored results)" : "")
ok || error("Reconciliation FAILED — refusing to continue (same gate as optimise_control_pulse_from_jld2).")


## 7. Linear seed fit

`fit_linear_seed` always uses `fit_mode=:linear` on the control I/Q from step 4 (`PULSE_CONFIG` if it parsed, otherwise the CSV).


In [ ]:
pulse_fit, u_fit, fit_report, segments = fit_linear_seed(
    ref;
    param_budget = pipe.param_budget,
    degree     = opt.degree,
    taper_frac = opt.taper_frac,
    verbose    = true,
)

println()
println("=== Fitted CompositePulse ===")
@printf("  k=%d  n_coeff_A=%d  n_coeff_f=%d  n_params=%d  (%d segments detected)\n",
    pulse_fit.k, pulse_fit.n_coeff_A, pulse_fit.n_coeff_f, n_params(pulse_fit), length(segments))

println()
println("=== Fit error ===")
@printf("  envelope (full I+iQ, rel L2)        = %.6g\n", fit_report.rel_l2_complex)
@printf("  Rabi amplitude (|E|, rel L2)         = %.6g\n", fit_report.rel_l2_A)
@printf("  instantaneous frequency (rel L2)     = %.6g\n", fit_report.rel_l2_f)
@printf("  instantaneous frequency (phase RMS)  = %.6g rad\n", fit_report.phi_rms_rad)
@printf("  amplitude coeffs floored / clipped   = %d / %d\n", fit_report.n_cA_floored, fit_report.n_cf_clipped)


## 8. Optimise on the same reference configs (`pulse_optimizer2.jl`)

Warm-starts `optimise_composite_pulse` from the fitted seed. Every optimiser knob is `jld2_optimizer_defaults()`, plus `signal_E_of_t = ref.signal_E_of_t`.


In [ ]:
println("=== optimiser kwargs (jld2_optimizer_defaults) ===")
for (k, v) in pairs(opt)
    @printf("  %-22s = %s\n", k, repr(v))
end
println("  signal_E_of_t         = ref.signal_E_of_t  (USE_SIGNAL=$(ref.use_signal))")
println("  warm_start_u          = u_fit")
println()

global_best_u, global_best_cost, opt_pulse, u0, initial_metrics, history, final_metrics, optimizer_settings = optimise_composite_pulse(
    pulse_fit.k, pulse_fit.n_coeff_A, pulse_fit.n_coeff_f, d;
    opt...,
    warm_start_u = u_fit,
    signal_E_of_t = ref.signal_E_of_t,
)

println()
println("=== Optimisation result ===")
println("  initial: cost=$(round(initial_metrics[1]; digits=4))  inversion=$(round(initial_metrics[2]; digits=4))  silencing=$(round(initial_metrics[3]; digits=4))  duration=$(round(initial_metrics[4]; digits=6))  coherence=$(round(initial_metrics[5]; digits=4))")
println("  final:   cost=$(round(final_metrics[1]; digits=4))  inversion=$(round(final_metrics[2]; digits=4))  silencing=$(round(final_metrics[3]; digits=4))  duration=$(round(final_metrics[4]; digits=6))  coherence=$(round(final_metrics[5]; digits=4))")
println("  global_best_cost returned = $(round(global_best_cost; digits=4))")
